In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import matplotlib.pyplot as plt
import copy
import random
from collections import defaultdict, Counter
import time

from sklearn.cluster import KMeans
from utils import RandomForestClassifierUnc, XGBEnsemble
from sklearn.utils import resample
from sklearn.base import clone

sns.set_style("whitegrid")

import matplotlib

# set font size to 16
matplotlib.rcParams.update({"font.size": 16})

SEED = 4

In [2]:
# the initial 80-20 stratified split to form test set and full training set
def initial_split(df_dropped):
    # define train set and test set for each 
    X_train_files = []
    X_test_files = []
    y_train_files = []
    y_test_files = []

    cols = df_dropped.columns

    X_file = (df_dropped.drop(["Label", "task_id"], axis=1).to_numpy())
    y_file = df_dropped["Label"].to_numpy()
    # 80-20 stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X_file, y_file, test_size=0.2, stratify=y_file)

    X_train_files.append(X_train)
    X_test_files.append(X_test)
    y_train_files.append(y_train)
    y_test_files.append(y_test)
        
    return X_train_files, X_test_files, y_train_files, y_test_files


In [3]:
# The static ML-based upper benchmark trained on full training set
def full_file_train(X_train_files, X_test_files, y_train_files, y_test_files, benchmark_dict):
    
    full_file_train_data = copy.deepcopy(X_train_files[0])
    full_file_train_labels = copy.deepcopy(y_train_files[0])
    full_file_test_data = copy.deepcopy(X_test_files[0])
    full_file_test_labels = copy.deepcopy(y_test_files[0])
    
    # Change the y_labels into 'malicious'/'benign' for all y files
    for i in range(len(full_file_train_labels)):
        if full_file_train_labels[i] != 'BENIGN':
            full_file_train_labels[i] = 'Malicious'
    for i in range(len(full_file_test_labels)):
        if full_file_test_labels[i] != 'BENIGN':
            full_file_test_labels[i] = 'Malicious'

    full_model = RandomForestClassifier(n_estimators=10, n_jobs=-1)
    full_model.fit(full_file_train_data, full_file_train_labels)
    model_preds = full_model.predict(full_file_test_data)
    
    benchmark_dict["Benchmark_Accuracy"].append(accuracy_score(full_file_test_labels, model_preds))
    benchmark_dict["Benchmark_Precision"].append(precision_score(full_file_test_labels, model_preds, pos_label = 'Malicious'))
    benchmark_dict["Benchmark_Recall"].append(recall_score(full_file_test_labels, model_preds, pos_label = 'Malicious'))
    benchmark_dict["Benchmark_F1score"].append(f1_score(full_file_test_labels, model_preds, pos_label = 'Malicious'))
    
    return benchmark_dict

In [4]:
#Simulation loop for PROACT
def PROACT(X_train_files_init, y_train_files_init, X_test_files_init, y_test_files_init, 
                       start_type, accuracy_dict, precision_dict, recall_dict, f1score_dict, distributions_dict, time_dict, run_n):
    #Table to discriminate the variables of 65 strategies
    #1 : I-LCS
    #2_batch: B-LCS
    #3_rand: RS
    #4_ku: I-KUS
    #5_ku_batch: B-KUS
    #6_qbc: QBC

    max_loop = 399 # stops when training set size = 400
    
    X_train_files = copy.deepcopy(X_train_files_init[0])
    y_train_files = copy.deepcopy(y_train_files_init[0])
    X_test_files = copy.deepcopy(X_test_files_init[0])
    y_test_files = copy.deepcopy(y_test_files_init[0])

    dataset_size_single = []
    dataset_size_batch = []

    #Formation of initial training set 
    # Keep only BENIGN and start_type
    mask = np.isin(y_train_files, [start_type, "BENIGN"])
    X_filtered, y_filtered = X_train_files[mask], y_train_files[mask]

    # Separate per class
    benign_idx = np.where(y_filtered == "BENIGN")[0]
    attack_idx = np.where(y_filtered == start_type)[0]

    # Pick exactly 25 from each class using StratifiedShuffleSplit
    sss = StratifiedShuffleSplit(n_splits=1, train_size=25)

    # BENIGN
    benign_selected, _ = next(sss.split(X_filtered[benign_idx], y_filtered[benign_idx]))
    benign_selected = benign_idx[benign_selected]

    # start_type
    attack_selected, _ = next(sss.split(X_filtered[attack_idx], y_filtered[attack_idx]))
    attack_selected = attack_idx[attack_selected]

    # Combine selected indices
    train_idx = np.concatenate([benign_selected, attack_selected])

    # Initial labeled training set (balanced: 25+25 = 50 samples)
    X_train_al = X_filtered[train_idx].tolist()
    y_train_al = y_filtered[train_idx].tolist()

    # Build mask in original array to remove selected samples
    selected_mask = np.zeros(len(y_train_files), dtype=bool)
    selected_mask[np.where(mask)[0][train_idx]] = True

    # Remove selected while preserving original order
    X_train_files = X_train_files[~selected_mask]
    y_train_files = y_train_files[~selected_mask]

    al_dataset_size = len(X_train_al)
    
    #For tracking the distribution of traffic types
    y_train_types = copy.deepcopy(y_train_al)
    y_holdout_types = copy.deepcopy(y_train_files)
    y_train_types_batch = copy.deepcopy(y_train_al)
    y_holdout_types_batch = copy.deepcopy(y_train_files)
    y_train_types_rand = copy.deepcopy(y_train_al)
    y_holdout_types_rand = copy.deepcopy(y_train_files)
    y_train_types_ku = copy.deepcopy(y_train_al)
    y_holdout_types_ku = copy.deepcopy(y_train_files)
    y_train_types_kubatch = copy.deepcopy(y_train_al)
    y_holdout_types_kubatch = copy.deepcopy(y_train_files)
    y_train_types_qbc = copy.deepcopy(y_train_al)
    y_holdout_types_qbc = copy.deepcopy(y_train_files)

    # Change the y_labels into 'malicious'/'benign' for all y files
    #1: I-LCS
    for i in range(len(y_train_al)):
        if y_train_al[i] != 'BENIGN':
            y_train_al[i] = 'Malicious'
    for i in range(len(y_train_files)):
        if y_train_files[i]!= 'BENIGN':
            y_train_files[i] = 'Malicious'
    for i in range(len(y_test_files)):
        if y_test_files[i] != 'BENIGN':
            y_test_files[i] = 'Malicious'

    #2: B-LCS
    X_train_al_batch = copy.deepcopy(X_train_al)
    y_train_al_batch = copy.deepcopy(y_train_al)
    X_train_files_batch = copy.deepcopy(X_train_files)
    y_train_files_batch = copy.deepcopy(y_train_files)
    al_dataset_size_batch = len(X_train_al_batch)

    #3: RS
    X_train_al_rand = copy.deepcopy(X_train_al)
    y_train_al_rand = copy.deepcopy(y_train_al)
    X_train_files_rand = copy.deepcopy(X_train_files)
    y_train_files_rand = copy.deepcopy(y_train_files)
    al_dataset_size_rand = len(X_train_al_rand)

    #4: I-KUS
    X_train_al_ku = copy.deepcopy(X_train_al)
    y_train_al_ku = copy.deepcopy(y_train_al)
    X_train_files_ku = copy.deepcopy(X_train_files)
    y_train_files_ku = copy.deepcopy(y_train_files)
    al_dataset_size_ku = len(X_train_al_ku)

    #5: B-KUS
    X_train_al_kubatch = copy.deepcopy(X_train_al)
    y_train_al_kubatch = copy.deepcopy(y_train_al)
    X_train_files_kubatch = copy.deepcopy(X_train_files)
    y_train_files_kubatch = copy.deepcopy(y_train_files)
    al_dataset_size_kubatch = len(X_train_al_kubatch)

    #6: QBC
    X_train_al_qbc = copy.deepcopy(X_train_al)
    y_train_al_qbc = copy.deepcopy(y_train_al)
    X_train_files_qbc = copy.deepcopy(X_train_files)
    y_train_files_qbc = copy.deepcopy(y_train_files)
    al_dataset_size_qbc = len(X_train_al_qbc)

    #Initial training of ML model
    #1: I-LCS
    model = RandomForestClassifier(n_estimators=10, n_jobs=-1)
    model.fit(X_train_al, y_train_al)
    #2: B-LCS
    model_batch = RandomForestClassifier(n_estimators=10, n_jobs=-1)
    model_batch.fit(X_train_al_batch, y_train_al_batch)
    #3: RS
    model_rand = RandomForestClassifier(n_estimators=10, n_jobs=-1)
    model_rand.fit(X_train_al_rand, y_train_al_rand)
    #4: I-KUS
    model_ku = RandomForestClassifierUnc(n_estimators=10, n_jobs=-1)
    model_ku.fit(X_train_al_ku, y_train_al_ku)
    #5: B-KUS
    model_kubatch = RandomForestClassifierUnc(n_estimators=10, n_jobs=-1)
    model_kubatch.fit(X_train_al_kubatch, y_train_al_kubatch)
    #6: QBC
    model_qbc = RandomForestClassifier(n_estimators=10, n_jobs=-1)
    model_qbc.fit(X_train_al_qbc, y_train_al_qbc)

    #1: Compute f1-score on test set
    y_pred = model.predict(X_test_files)
    accuracy_dict[run_n]["Predictproba_single"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["Predictproba_single"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["Predictproba_single"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["Predictproba_single"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
    dataset_size_single.append(al_dataset_size)

    #2: compute f1-score on test set
    y_pred = model_batch.predict(X_test_files)
    accuracy_dict[run_n]["Predictproba_batch"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["Predictproba_batch"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["Predictproba_batch"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["Predictproba_batch"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
    dataset_size_batch.append(al_dataset_size_batch)

    #3: compute f1-score on test set
    y_pred = model_rand.predict(X_test_files)
    accuracy_dict[run_n]["Random"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["Random"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["Random"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["Random"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))

    #4: compute f1-score on test set
    y_pred = model_ku.predict(X_test_files)
    accuracy_dict[run_n]["KU_single"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["KU_single"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["KU_single"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["KU_single"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))

    #5: compute f1-score on test set
    y_pred = model_kubatch.predict(X_test_files)
    accuracy_dict[run_n]["KU_batch"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["KU_batch"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["KU_batch"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["KU_batch"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))

    #6: compute f1-score on test set
    y_pred = model_qbc.predict(X_test_files)
    accuracy_dict[run_n]["QBC"].append(accuracy_score(y_test_files, y_pred))
    precision_dict[run_n]["QBC"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
    recall_dict[run_n]["QBC"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
    f1score_dict[run_n]["QBC"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
    
    # Define your 8 labels explicitly
    all_labels = ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]

    print("I-LCS")
    while len(X_train_files) > 0 and al_dataset_size <= max_loop:

        start_selection = time.time()
        probas = model.predict_proba(X_train_files)

        # Find the index of (min of max)
        row_maxes = np.max(probas, axis=1) 
        min_val = np.min(row_maxes)
        indices = np.where(row_maxes == min_val)[0]
        rndm_idx = random.randint(0, len(indices)-1)
        min_idx = indices[rndm_idx]

        X_train_al.append(X_train_files[min_idx])
        y_train_al.append(y_train_files[min_idx])
        end_selection = time.time()

        # Calculate selection time
        selection_time = end_selection - start_selection

        # Retrain
        start_train = time.time()
        model.fit(X_train_al, y_train_al)
        end_train = time.time()

        # Calculate train time
        train_time = end_train - start_train

        #For type percentages
        y_train_types.append(y_holdout_types[min_idx])
        y_holdout_types = np.delete(y_holdout_types, min_idx, axis=0)

        #Compute the distributions of all types
        counts = Counter(label for label in y_train_types[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["Predictproba_single"].append(percentages)

        #Extract the train set
        X_train_files = np.delete(X_train_files, min_idx, axis=0)
        y_train_files = np.delete(y_train_files, min_idx, axis=0)

        # compute f1-score on test set
        y_pred = model.predict(X_test_files)
        accuracy_dict[run_n]["Predictproba_single"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["Predictproba_single"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["Predictproba_single"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["Predictproba_single"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["Predictproba_single"].append([selection_time, train_time])
        al_dataset_size += 1
        dataset_size_single.append(al_dataset_size)

    print("B-LCS")
    while len(X_train_files_batch) > 10 and al_dataset_size_batch <= (max_loop - 9):

        start_selection = time.time()
        probas_batch = model_batch.predict_proba(X_train_files_batch)

        # Find the index of (min of max)
        row_maxes = np.max(probas_batch, axis=1) 
        min_10idx = np.argsort(row_maxes)[:10]

        # Retrain
        X_train_al_batch.extend(X_train_files_batch[min_10idx])
        y_train_al_batch.extend(y_train_files_batch[min_10idx])
        end_selection = time.time()

        #Calculate selection Time
        selection_time = end_selection - start_selection

        start_train = time.time()
        model_batch.fit(X_train_al_batch, y_train_al_batch)
        end_train = time.time()

        #Calculate train time
        train_time = end_train - start_train

        #For type percentages
        y_train_types_batch.extend(y_holdout_types_batch[min_10idx])
        y_holdout_types_batch = np.delete(y_holdout_types_batch, min_10idx, axis=0)

        #Extract the train set
        X_train_files_batch = np.delete(X_train_files_batch, min_10idx, axis=0)
        y_train_files_batch = np.delete(y_train_files_batch, min_10idx, axis=0)

        counts = Counter(label for label in y_train_types_batch[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["Predictproba_batch"].append(percentages)

        # compute f1-score on test set
        y_pred = model_batch.predict(X_test_files)
        accuracy_dict[run_n]["Predictproba_batch"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["Predictproba_batch"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["Predictproba_batch"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["Predictproba_batch"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["Predictproba_batch"].append([selection_time, train_time])
        al_dataset_size_batch += 10
        dataset_size_batch.append(al_dataset_size_batch)

    print("RS")
    while len(X_train_files_rand) > 0 and al_dataset_size_rand <= max_loop:

        start_selection = time.time()        
        random_idx = random.randint(0, len(X_train_files_rand)-1)

        # Retrain
        X_train_al_rand.append(X_train_files_rand[random_idx])
        y_train_al_rand.append(y_train_files_rand[random_idx])
        end_selection = time.time()

        #Calculate selection time 
        selection_time = end_selection - start_selection

        start_train = time.time()
        model_rand.fit(X_train_al_rand, y_train_al_rand)
        end_train = time.time()

        #Calculate train time
        train_time = end_train - start_train

        #For type percentages
        y_train_types_rand.append(y_holdout_types_rand[random_idx])
        y_holdout_types_rand = np.delete(y_holdout_types_rand, random_idx, axis=0)

        #Extract the train set
        X_train_files_rand = np.delete(X_train_files_rand, random_idx, axis=0)
        y_train_files_rand = np.delete(y_train_files_rand, random_idx, axis=0)

        counts = Counter(label for label in y_train_types_rand[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["Random"].append(percentages)

        # compute f1-score on test set
        y_pred = model_rand.predict(X_test_files)
        accuracy_dict[run_n]["Random"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["Random"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["Random"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["Random"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["Random"].append([selection_time, train_time])
        al_dataset_size_rand += 1

    print("I-KUS")
    while len(X_train_files_ku) > 0 and al_dataset_size_ku <= max_loop:

        start_selection = time.time()
        _, ku = model_ku.predict_uncertainty(X_train_files_ku)

        max_val = np.max(ku)
        indices = np.where(ku == max_val)[0]
        rndm_idx = random.randint(0, len(indices)-1)
        idx = indices[rndm_idx]

        #idx = np.argmax(ku)
        X_train_al_ku.append(X_train_files_ku[idx])
        y_train_al_ku.append(y_train_files_ku[idx])
        end_selection = time.time()

        #Calculate selection time
        selection_time = end_selection - start_selection

        start_train = time.time()
        model_ku = model_ku.fit(X_train_al_ku, y_train_al_ku)
        end_train = time.time()

        #Calculate train time
        train_time = end_train - start_train

        X_train_files_ku = np.delete(X_train_files_ku, idx, axis=0)
        y_train_files_ku = np.delete(y_train_files_ku, idx, axis=0)

        #For type percentages
        y_train_types_ku.append(y_holdout_types_ku[idx])
        y_holdout_types_ku = np.delete(y_holdout_types_ku, idx, axis=0)

        counts = Counter(label for label in y_train_types_ku[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["KU_single"].append(percentages)

        y_pred = model_ku.predict(X_test_files)
        accuracy_dict[run_n]["KU_single"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["KU_single"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["KU_single"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["KU_single"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["KU_single"].append([selection_time, train_time])
        al_dataset_size_ku += 1

    print("B-KUS")
    while len(X_train_files_kubatch) > 0 and al_dataset_size_kubatch <= (max_loop - 9):
       
        start_selection = time.time()
        _, ku = model_kubatch.predict_uncertainty(X_train_files_kubatch)

        max_10idx = np.argsort(ku)[::-1][:10]

        # Retrain
        X_train_al_kubatch.extend(X_train_files_kubatch[max_10idx])
        y_train_al_kubatch.extend(y_train_files_kubatch[max_10idx])
        end_selection = time.time()

        #Calculate selection time
        selection_time = end_selection - start_selection

        start_train = time.time()
        model_kubatch.fit(X_train_al_kubatch, y_train_al_kubatch)
        end_train = time.time()

        #Calculate train time
        train_time = end_train - start_train

        #For type percentages
        y_train_types_kubatch.extend(y_holdout_types_kubatch[max_10idx])
        y_holdout_types_kubatch = np.delete(y_holdout_types_kubatch, max_10idx, axis=0)

        #Extract the train set
        X_train_files_kubatch = np.delete(X_train_files_kubatch, max_10idx, axis=0)
        y_train_files_kubatch = np.delete(y_train_files_kubatch, max_10idx, axis=0)

        counts = Counter(label for label in y_train_types_kubatch[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["KU_batch"].append(percentages)

        y_pred = model_kubatch.predict(X_test_files)
        accuracy_dict[run_n]["KU_batch"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["KU_batch"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["KU_batch"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["KU_batch"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["KU_batch"].append([selection_time, train_time])
        al_dataset_size_kubatch += 10

    print("QBC")
    n_committee = 5
    while len(X_train_files_qbc) > 0 and al_dataset_size_qbc <= max_loop:

        # Build Committee
        start_selection = time.time()
        committee = []
        for i in range(n_committee):
            X_boot, y_boot = resample(X_train_al_qbc, y_train_al_qbc, replace=True)
            model = clone(model_qbc)
            model.fit(X_boot, y_boot)
            committee.append(model)

        # Committee Predictions
        all_preds = np.array([model.predict(X_train_files_qbc) for model in committee])  # shape: (n_models, n_samples)

        #Compute Disagreement
        # Find high entropy
        vote_entropy = []
        for i in range(all_preds.shape[1]):  # for each sample
            votes = Counter(all_preds[:, i])
            probs = np.array(list(votes.values())) / n_committee
            entropy = -np.sum(probs * np.log2(probs + 1e-9))
            vote_entropy.append(entropy)
        vote_entropy = np.array(vote_entropy)

        # Select most uncertain sample
        max_val = np.max(vote_entropy) 
        indices = np.where(vote_entropy == max_val)[0]
        max_idx = random.choice(indices)

        X_train_al_qbc.append(X_train_files_qbc[max_idx])
        y_train_al_qbc.append(y_train_files_qbc[max_idx])
        end_selection = time.time()

        #Calculate selection time
        selection_time = end_selection - start_selection

        start_train = time.time()
        model_qbc.fit(X_train_al_qbc, y_train_al_qbc)
        end_train = time.time()

        #Calculate train time
        train_time = end_train - start_train

        #For type percentages
        y_train_types_qbc.append(y_holdout_types_qbc[max_idx])
        y_holdout_types_qbc = np.delete(y_holdout_types_qbc, max_idx, axis=0)

        counts = Counter(label for label in y_train_types_qbc[50:])
        total = sum(counts.values())
        # Compute percentage for each label in predefined order
        percentages = [(counts.get(label, 0) / total) * 100 if total > 0 else 0.0 for label in all_labels]
        distributions_dict[run_n]["QBC"].append(percentages)
        
        # Remove selected sample from pool
        X_train_files_qbc = np.delete(X_train_files_qbc, max_idx, axis=0)
        y_train_files_qbc = np.delete(y_train_files_qbc, max_idx, axis=0)

        y_pred = model_qbc.predict(X_test_files)
        accuracy_dict[run_n]["QBC"].append(accuracy_score(y_test_files, y_pred))
        precision_dict[run_n]["QBC"].append(precision_score(y_test_files, y_pred, pos_label= "Malicious"))
        recall_dict[run_n]["QBC"].append(recall_score(y_test_files, y_pred, pos_label= "Malicious"))
        f1score_dict[run_n]["QBC"].append(f1_score(y_test_files, y_pred, pos_label= "Malicious"))
        time_dict[run_n]["QBC"].append([selection_time, train_time])
        al_dataset_size_qbc += 1


    return (dataset_size_single, dataset_size_batch, accuracy_dict, precision_dict, recall_dict, f1score_dict, distributions_dict, time_dict)


In [5]:
#Save accuracy, precision, recall and f1 score
def save_nested_dict_to_csv(nested_dict, filename):
    """
    Saves a nested dictionary (e.g., accuracy_dict[run][method] = list_of_values)
    into a single CSV file.
    Each row contains: Run, Method, Iteration, Value.
    """
    rows = []
    for run_n, methods in nested_dict.items():
        for method_name, values in methods.items():
            for i, val in enumerate(values):
                rows.append({
                    'Run': run_n,
                    'Method': method_name,
                    'Iteration': i,
                    'Value': val
                })
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"✅ Saved {filename} ({len(df)} rows)")

#Save time measurements
def save_time_dict_to_csv(time_dict, filename):
    """
    Saves time_dict[run][method] = [[selection_time, train_time], ...]
    into a single CSV file.
    Each row contains: Run, Method, Iteration, Selection_Time, Train_Time.
    """
    rows = []
    for run_n, methods in time_dict.items():
        for method_name, time_list in methods.items():
            for i, pair in enumerate(time_list):
                selection_time, train_time = pair
                rows.append({
                    'Run': run_n,
                    'Method': method_name,
                    'Iteration': i,
                    'Selection_Time': selection_time,
                    'Train_Time': train_time
                })
    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"✅ Saved {filename} ({len(df)} rows)")

#Save type distributions
def save_distributions_dict_to_csv(distributions_dict, all_labels, filename):
    """
    Saves distributions_dict[run][method] = [ [p_label1, p_label2, ...], ... ]
    into a single CSV file.
    Each row contains: Run, Method, Iteration, and one column per label.
    
    Parameters:
    - distributions_dict: nested dict
    - all_labels: list of label names in the same order as the inner arrays
    - filename: output CSV filename
    """
    rows = []
    for run_n, methods in distributions_dict.items():
        for method_name, dist_list in methods.items():
            for i, percentages in enumerate(dist_list):
                row = {
                    'Run': run_n,
                    'Method': method_name,
                    'Iteration': i
                }
                # Add each label’s percentage as a separate column
                for label, pct in zip(all_labels, percentages):
                    row[label] = pct
                rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"✅ Saved {filename} ({len(df)} rows)")

In [7]:
n_runs = 100
df_dropped = pd.read_csv("../CIC_dfdropped/03-11/df_dropped_3000.csv")
#Create dictionaries to store results
#all benchmarks for n_runs
benchmark_dict = defaultdict(list) #benchmark_dict["benchmart_metric"]
#type distributions for n_runs and 6 algos from dataset_size 50:400
distributions_dict = defaultdict(lambda: defaultdict(list)) #distributions_dict[n_runs]["algo"] = [benign, portmap, netbios, ..., syn]
#accuracy for n_runs and 6 algos from dataset_size 50:400
accuracy_dict = defaultdict(lambda: defaultdict(list)) #accuracy_dict[n_runs]["algo"]
#precision for n_runs and 6 algos from dataset_size 50:400
precision_dict = defaultdict(lambda: defaultdict(list)) #precision_dict[n_runs]["algo"]
#recall for n_runs and 6 algos from dataset_size 50:400
recall_dict = defaultdict(lambda: defaultdict(list)) #recall_dict[n_runs]["algo"]
#f1-score for n_runs and 6 algos from dataset_size 50:400
f1score_dict = defaultdict(lambda: defaultdict(list)) #f1score_dict[n_runs]["algo"]
#selection and train time for n_runs and 6 algos from dataset_size 50:400
time_dict = defaultdict(lambda: defaultdict(list)) #time_dict[n_runs]["algo"] = [selection_time, train_time]


for n in range(n_runs):
    print(n)
    #initial train test split
    X_train_files, X_test_files, y_train_files, y_test_files = initial_split(df_dropped)

    #upper-benchmark analysis: ML-based
    benchmark_dict = full_file_train(X_train_files, X_test_files, y_train_files, y_test_files, benchmark_dict)

    #active learning: PROACT
    #types: "BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "Syn"
    start_type = "Portmap"
    (dataset_size_single, dataset_size_batch, accuracy_dict, precision_dict, recall_dict, f1score_dict, 
            distributions_dict, time_dict) = PROACT(X_train_files, y_train_files,X_test_files, y_test_files, 
                start_type, accuracy_dict, precision_dict, recall_dict, f1score_dict, distributions_dict, time_dict, n)
    
#Save the dictionaries to csv files
df = pd.DataFrame(benchmark_dict)
df.to_csv(f"./results_dictionaries/benchmark_dict_{start_type}_{n_runs}.csv", index=False)
print(f"✅ Benchmarks Saved ")
save_nested_dict_to_csv(accuracy_dict, f"./results_dictionaries/accuracy_dict_{start_type}_{n_runs}.csv")
save_nested_dict_to_csv(precision_dict, f"./results_dictionaries/precision_dict_{start_type}_{n_runs}.csv")
save_nested_dict_to_csv(recall_dict, f"./results_dictionaries/recall_dict_{start_type}_{n_runs}.csv")
save_nested_dict_to_csv(f1score_dict, f"./results_dictionaries/f1score_dict_{start_type}_{n_runs}.csv")
save_time_dict_to_csv(time_dict, f"./results_dictionaries/time_dict_{start_type}_{n_runs}.csv")
all_labels = ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
save_distributions_dict_to_csv(distributions_dict, all_labels, f"./results_dictionaries/distributions_dict_{start_type}_{n_runs}.csv")


0
I-LCS
B-LCS
RS
I-KUS
B-KUS
QBC
1


KeyboardInterrupt: 

In [ ]:

#To use the csv files again (dictionaries)
#Load accuracy, precision, recall and f1 score
def load_dict_from_csv(filename):
    df = pd.read_csv(filename)
    nested_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        nested_dict.setdefault(run, {})[method] = group['Value'].tolist()
    return nested_dict

#Load time measurements
def load_time_dict_from_csv(filename):
    df = pd.read_csv(filename)
    time_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        pairs = group[['Selection_Time', 'Train_Time']].values.tolist()
        time_dict.setdefault(run, {})[method] = pairs
    return time_dict

#Load type distributions
def load_distributions_dict_from_csv(filename, all_labels):
    df = pd.read_csv(filename)
    dist_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        arrs = group[all_labels].values.tolist()
        dist_dict.setdefault(run, {})[method] = arrs
    return dist_dict

#Load benchmark values
def load_benchmark_dict_from_csv(filename):
    df = pd.read_csv(filename)
    return {col: df[col].dropna().tolist() for col in df.columns}

benchmark_dict_loaded = load_benchmark_dict_from_csv(f"./results_dict/benchmark_dict_{start_type}_{n_runs}.csv")
accuracy_dict_loaded = load_dict_from_csv( f"./results_dict/accuracy_dict_{start_type}_{n_runs}.csv")
precision_dict_loaded = load_dict_from_csv( f"./results_dict/precision_dict_{start_type}_{n_runs}.csv")
recall_dict_loaded = load_dict_from_csv( f"./results_dict/recall_dict_{start_type}_{n_runs}.csv")
f1score_dict_loaded = load_dict_from_csv( f"./results_dict/f1score_dict_{start_type}_{n_runs}.csv")
time_dict_loaded = load_time_dict_from_csv(f"./results_dict/time_dict_{start_type}_{n_runs}.csv")
all_labels = ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
distributions_dict_loaded = load_distributions_dict_from_csv(f"./results_dict/distributions_dict_{start_type}_{n_runs}.csv", all_labels)
